In [ ]:
# ============================================================
# Notebook 3 — Multi-Agent Pipeline
# Planner → Coder → Reviewer
# ============================================================

print("Notebook 3 — Multi-Agent Pipeline")
print("Model: meta-llama/Llama-3.1-8B-Instruct")

Notebook 3 — Multi-Agent Pipeline
Model: meta-llama/Llama-3.1-8B-Instruct


In [ ]:
# ============================================
# Cell 2: Imports
# ============================================

import os
import json
import time

print("✓ Imports loaded successfully.")

✓ Imports loaded successfully.


In [ ]:
# ============================================
# Cell 3: Model Configuration
# ============================================

MODEL = "meta-llama/Llama-3.1-8B-Instruct"

print("Model:", MODEL)

Model: meta-llama/Llama-3.1-8B-Instruct


In [ ]:
# ============================================
# Cell 4: Hugging Face Authentication
# ============================================

HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    print("✓ Hugging Face token found.")
else:
    print("⚠ Hugging Face token not found.")

⚠ Hugging Face token not found.


In [ ]:
# ============================================
# Cell 4: Hugging Face Authentication
# ============================================

from huggingface_hub import login

login()

print("✓ Hugging Face authentication completed.")

✓ Hugging Face authentication completed.


In [ ]:
# ============================================
# Cell 5: Hugging Face Inference Client
# ============================================

from huggingface_hub import InferenceClient

client = InferenceClient(
    model=MODEL
)

print("✓ Hugging Face inference client initialized.")
print("Model:", MODEL)

✓ Hugging Face inference client initialized.
Model: meta-llama/Llama-3.1-8B-Instruct


In [ ]:
# ============================================
# Cell 6: Test Model Connection
# ============================================

response = client.chat_completion(
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Connection successful."
        }
    ],
    max_tokens=20,
    temperature=0.01
)

print(response.choices[0].message.content)

Connection successful.


In [ ]:
# ============================================
# Cell 7: Recreate Final Core Tasks
# ============================================

import json
from datasets import load_dataset

print("Loading HumanEval dataset...")

ds = load_dataset(
    "openai/openai_humaneval",
    split="test"
)

print("✓ HumanEval loaded:", len(ds), "tasks")

Loading HumanEval dataset...


README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

✓ HumanEval loaded: 164 tasks


In [ ]:
# ============================================
# Cell 8: Build Final 30 Core Tasks
# ============================================

task_difficulty = {
    # EASY
    "HumanEval/53": "easy",
    "HumanEval/23": "easy",
    "HumanEval/45": "easy",
    "HumanEval/27": "easy",
    "HumanEval/15": "easy",
    "HumanEval/16": "easy",
    "HumanEval/98": "easy",
    "HumanEval/35": "easy",
    "HumanEval/13": "easy",
    "HumanEval/83": "easy",

    # MEDIUM
    "HumanEval/50": "medium",
    "HumanEval/58": "medium",
    "HumanEval/89": "medium",
    "HumanEval/26": "medium",
    "HumanEval/154": "medium",
    "HumanEval/140": "medium",
    "HumanEval/111": "medium",
    "HumanEval/93": "medium",
    "HumanEval/56": "medium",
    "HumanEval/31": "medium",

    # HARD
    "HumanEval/7": "hard",
    "HumanEval/25": "hard",
    "HumanEval/132": "hard",
    "HumanEval/54": "hard",
    "HumanEval/20": "hard",
    "HumanEval/6": "hard",
    "HumanEval/41": "hard",
    "HumanEval/126": "hard",
    "HumanEval/43": "hard",
    "HumanEval/99": "hard"
}

task_lookup = {
    item["task_id"]: item
    for item in ds
}

core_tasks = []

for task_id, difficulty in task_difficulty.items():

    task = task_lookup[task_id]

    core_tasks.append({
        "task_id": task["task_id"],
        "prompt": task["prompt"],
        "test": task["test"],
        "entry_point": task["entry_point"],
        "difficulty": difficulty
    })

print("✓ Final core task set recreated")
print("Total tasks:", len(core_tasks))

✓ Final core task set recreated
Total tasks: 30


In [ ]:
# ============================================
# Cell 9: Verify Core Tasks
# ============================================

from collections import Counter

counts = Counter(
    task["difficulty"]
    for task in core_tasks
)

print("Easy  :", counts["easy"])
print("Medium:", counts["medium"])
print("Hard  :", counts["hard"])
print("Total :", len(core_tasks))

Easy  : 10
Medium: 10
Hard  : 10
Total : 30


In [ ]:
# ============================================
# Cell 10: Save Core Tasks
# ============================================

with open("core_tasks.json", "w") as f:
    json.dump(core_tasks, f, indent=2)

print("✓ core_tasks.json created in Notebook 3")

✓ core_tasks.json created in Notebook 3


In [ ]:
# ============================================
# Cell 11: Planner Agent
# ============================================

PLANNER_SYSTEM_PROMPT = """
You are the Planner agent in a multi-agent Python code-generation system.

You will receive a HumanEval programming problem.

Your task is to analyze the problem and produce a concise implementation plan
for another coding agent.

Requirements:
1. Understand the required function and its expected behavior.
2. Identify important inputs and outputs.
3. Identify edge cases and constraints.
4. Describe the algorithm or implementation steps clearly.
5. You may provide pseudocode.
6. Do NOT write the final Python implementation.
7. Do NOT use Markdown code fences for the plan.
"""

def planner_agent(task):
    """
    Planner:
    HumanEval problem → implementation plan
    """

    plan, latency, tokens, token_source = call_llm(
        system_prompt=PLANNER_SYSTEM_PROMPT,
        user_prompt=task["prompt"],
        max_tokens=1024
    )

    return {
        "task_id": task["task_id"],
        "plan": plan,
        "latency_sec": latency,
        "tokens": tokens,
        "token_source": token_source
    }


print("✓ Planner agent defined successfully.")

✓ Planner agent defined successfully.


In [ ]:
# ============================================
# Cell 12: Coder Agent
# ============================================

CODER_SYSTEM_PROMPT = """
You are the Coder agent in a multi-agent Python code-generation system.

You will receive:
1. A HumanEval programming problem.
2. An implementation plan created by a Planner agent.

Your task is to implement the required function.

Requirements:
1. Follow the HumanEval problem specification.
2. Use the Planner's analysis as guidance.
3. Keep the required function name and signature unchanged.
4. Return only executable Python code.
5. Do not include explanations.
6. Do not include Markdown code fences.
7. Do not modify the problem specification.
"""

def coder_agent(task, plan):

    coder_prompt = f"""
HumanEval Problem:

{task["prompt"]}

Planner's Implementation Plan:

{plan}
"""

    code, latency, tokens, token_source = call_llm(
        system_prompt=CODER_SYSTEM_PROMPT,
        user_prompt=coder_prompt,
        max_tokens=1024
    )

    return {
        "task_id": task["task_id"],
        "generated_code": code,
        "latency_sec": latency,
        "tokens": tokens,
        "token_source": token_source
    }


print("✓ Coder agent defined successfully.")

✓ Coder agent defined successfully.


In [ ]:
# ============================================
# Cell 13: Reviewer Agent
# ============================================

REVIEWER_SYSTEM_PROMPT = """
You are the Reviewer agent in a multi-agent Python code-generation system.

You will receive:
1. A HumanEval programming problem.
2. A Python implementation produced by a Coder agent.

Your task is to review the implementation and produce the final corrected code.

Requirements:
1. Check whether the implementation satisfies the problem.
2. Check the function name and signature.
3. Check logical correctness.
4. Check important edge cases.
5. Correct any problems you identify.
6. Return the COMPLETE final Python implementation.
7. Return only executable Python code.
8. Do not include explanations.
9. Do not include Markdown code fences.
"""

def reviewer_agent(task, coder_code):

    reviewer_prompt = f"""
HumanEval Problem:

{task["prompt"]}

Coder's Implementation:

{coder_code}
"""

    final_code, latency, tokens, token_source = call_llm(
        system_prompt=REVIEWER_SYSTEM_PROMPT,
        user_prompt=reviewer_prompt,
        max_tokens=1024
    )

    return {
        "task_id": task["task_id"],
        "reviewer_final": final_code,
        "latency_sec": latency,
        "tokens": tokens,
        "token_source": token_source
    }


print("✓ Reviewer agent defined successfully.")

✓ Reviewer agent defined successfully.


In [ ]:
# ============================================
# Cell 14: Pipeline B — Multi-Agent
# ============================================

def multi_agent_pipeline(task):
    """
    Pipeline B:

    HumanEval Task
          ↓
       Planner
          ↓
        Coder
          ↓
       Reviewer
          ↓
      Final Code
    """

    # -------------------------
    # 1. Planner
    # -------------------------

    planner_result = planner_agent(task)

    # -------------------------
    # 2. Coder
    # -------------------------

    coder_result = coder_agent(
        task,
        planner_result["plan"]
    )

    # -------------------------
    # 3. Reviewer
    # -------------------------

    reviewer_result = reviewer_agent(
        task,
        coder_result["generated_code"]
    )

    # -------------------------
    # Total metrics
    # -------------------------

    total_latency = (
        planner_result["latency_sec"]
        + coder_result["latency_sec"]
        + reviewer_result["latency_sec"]
    )

    total_tokens = (
        planner_result["tokens"]
        + coder_result["tokens"]
        + reviewer_result["tokens"]
    )

    sources = [
        planner_result["token_source"],
        coder_result["token_source"],
        reviewer_result["token_source"]
    ]

    token_source = (
        "api"
        if all(source == "api" for source in sources)
        else "estimated"
    )

    # -------------------------
    # Store everything
    # -------------------------

    result = {
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],

        "plan": planner_result["plan"],

        # IMPORTANT:
        # Keep Coder and Reviewer outputs separately
        "coder_draft": coder_result["generated_code"],
        "reviewer_final": reviewer_result["reviewer_final"],

        "planner_latency_sec": planner_result["latency_sec"],
        "coder_latency_sec": coder_result["latency_sec"],
        "reviewer_latency_sec": reviewer_result["latency_sec"],

        "latency_sec": total_latency,
        "tokens": total_tokens,
        "token_source": token_source
    }

    return result


print("✓ Pipeline B — Planner → Coder → Reviewer defined successfully.")

✓ Pipeline B — Planner → Coder → Reviewer defined successfully.


In [ ]:
# ============================================
# Cell 15: Define call_llm()
# ============================================

import time

def call_llm(system_prompt, user_prompt, max_tokens=1024):

    start_time = time.time()

    response = client.chat_completion(
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        max_tokens=max_tokens,
        temperature=0.01
    )

    latency = time.time() - start_time

    output = response.choices[0].message.content

    # API token usage
    tokens = None

    if hasattr(response, "usage") and response.usage is not None:
        if hasattr(response.usage, "total_tokens"):
            tokens = response.usage.total_tokens

    # Fallback if API usage is unavailable
    if tokens is not None:
        token_source = "api"
    else:
        tokens = len(output.split())
        token_source = "estimated"

    return output, latency, tokens, token_source


print("✓ call_llm() defined successfully.")

✓ call_llm() defined successfully.


In [ ]:
test_task = next(
    task for task in core_tasks
    if task["task_id"] == "HumanEval/53"
)

print("Testing Pipeline B")
print("Task ID:", test_task["task_id"])
print("Difficulty:", test_task["difficulty"])
print("-" * 60)

pipeline_b_result = multi_agent_pipeline(test_task)

print("\n✓ Pipeline B smoke test completed")
print("Task ID:", pipeline_b_result["task_id"])
print(
    "Total latency:",
    round(pipeline_b_result["latency_sec"], 2),
    "seconds"
)
print("Total tokens:", pipeline_b_result["tokens"])
print("Token source:", pipeline_b_result["token_source"])

print("\n" + "=" * 60)
print("PLANNER OUTPUT")
print("=" * 60)
print(pipeline_b_result["plan"])

print("\n" + "=" * 60)
print("CODER DRAFT")
print("=" * 60)
print(pipeline_b_result["coder_draft"])

print("\n" + "=" * 60)
print("REVIEWER FINAL")
print("=" * 60)
print(pipeline_b_result["reviewer_final"])

Testing Pipeline B
Task ID: HumanEval/53
Difficulty: easy
------------------------------------------------------------

✓ Pipeline B smoke test completed
Task ID: HumanEval/53
Total latency: 17.68 seconds
Total tokens: 1077
Token source: api

PLANNER OUTPUT
Implementation Plan:

**Function Name:** add
**Purpose:** To add two integers, x and y.
**Inputs:**
	* x: An integer to be added.
	* y: An integer to be added.
**Outputs:** The sum of x and y.
**Edge Cases and Constraints:**
	* Both x and y are integers.
	* The function should handle negative numbers.
**Algorithm:**
1. Take the two integers, x and y, as input.
2. Use the built-in addition operator (+) to add x and y.
3. Return the sum.

**Pseudocode:**
```
function add(x, y):
    sum = x + y
    return sum
```
**Notes:**
* The function is designed to handle basic addition of two integers.
* The function does not handle non-integer inputs, as per the problem statement.
* The function does not handle floating-point numbers or complex 

In [ ]:
# ============================================
# Cell 16: Pipeline B — 3-Task Smoke Test
# ============================================

smoke_task_ids = [
    "HumanEval/53",   # Easy
    "HumanEval/50",   # Medium
    "HumanEval/7"     # Hard
]

pipeline_b_smoke_results = []

for task_id in smoke_task_ids:

    task = next(
        task for task in core_tasks
        if task["task_id"] == task_id
    )

    print("=" * 60)
    print("Task:", task["task_id"])
    print("Difficulty:", task["difficulty"])
    print("-" * 60)

    result = multi_agent_pipeline(task)

    pipeline_b_smoke_results.append(result)

    print("Total latency:",
          round(result["latency_sec"], 2), "sec")
    print("Total tokens:", result["tokens"])
    print("Token source:", result["token_source"])

    print("\nCoder draft:")
    print(result["coder_draft"])

    print("\nReviewer final:")
    print(result["reviewer_final"])

print("\n✓ Pipeline B 3-task smoke test completed.")

Task: HumanEval/53
Difficulty: easy
------------------------------------------------------------
Total latency: 17.23 sec
Total tokens: 1145
Token source: api

Coder draft:
def add(x: int, y: int):
    sum = x + y
    return sum

Reviewer final:
The Coder's implementation is correct and satisfies the problem. The function name and signature are also correct. The implementation is logically correct and handles the edge case where the inputs are integers.

Here is the final corrected code:

```python
def add(x: int, y: int):
    """Add two numbers x and y
    >>> add(2, 3)
    5
    >>> add(5, 7)
    12
    """
    return x + y
```
Task: HumanEval/50
Difficulty: medium
------------------------------------------------------------
Total latency: 28.16 sec
Total tokens: 1845
Token source: api

Coder draft:
def decode_shift(s: str):
    """
    takes as input string encoded with encode_shift function. Returns decoded string.
    """
    return "".join([chr(((ord(ch) - 5) % 26) + ord("a")) fo

In [ ]:
# ============================================
# PIPELINE B — HUGGING FACE SETUP
# ============================================

from google.colab import userdata
from huggingface_hub import InferenceClient

HF_TOKEN_B = userdata.get("SecureAgentEvalExp")

if not HF_TOKEN_B:
    raise RuntimeError(
        "SecureAgentEvalExp not found in Colab Secrets."
    )

HF_TOKEN_B = HF_TOKEN_B.strip()

print("Token loaded:", bool(HF_TOKEN_B))
print("Starts with hf_:", HF_TOKEN_B.startswith("hf_"))
print("Token length:", len(HF_TOKEN_B))

Token loaded: True
Starts with hf_: True
Token length: 37


In [ ]:
# ============================================
# PIPELINE B — CLIENT INITIALIZATION
# ============================================

MODEL = "meta-llama/Llama-3.1-8B-Instruct"

client_b = InferenceClient(
    model=MODEL,
    token=HF_TOKEN_B,
    provider="auto"
)

print("✓ Pipeline B client initialized")
print("Model:", MODEL)
print("Provider:", "auto")
print("Platform: Hugging Face")

✓ Pipeline B client initialized
Model: meta-llama/Llama-3.1-8B-Instruct
Provider: auto
Platform: Hugging Face


In [ ]:
# ============================================
# PIPELINE B — ONE-CALL SMOKE TEST
# ============================================

test_response_b = client_b.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": (
                "You are a Python programming assistant. "
                "Return only valid Python code."
            )
        },
        {
            "role": "user",
            "content": (
                "Write a Python function add(x, y) "
                "that returns x + y."
            )
        }
    ],
    temperature=0.01,
    max_tokens=100
)

test_code_b = test_response_b.choices[0].message.content

print("✓ Hugging Face inference successful")
print("\nResponse:")
print(test_code_b)

if test_response_b.usage:
    print("\nTokens:", test_response_b.usage.total_tokens)

✓ Hugging Face inference successful

Response:
Here is a Python function that meets the requirements:

```python
def add(x, y):
    """
    Returns the sum of x and y.
    
    Args:
        x (int or float): The first number to add.
        y (int or float): The second number to add.
    
    Returns:
        int or float: The sum of x and y.
    """
    return x + y
```

Example use cases:

```python
print(add(3, 5

Tokens: 143


In [ ]:
# ============================================
# PIPELINE B — FIND EXISTING HUMANEVAL DATASET
# ============================================

import os

print("Searching current workspace for HumanEval files...\n")

matches = []

for root, dirs, files in os.walk("."):
    # Don't search huge system/cache directories
    dirs[:] = [
        d for d in dirs
        if d not in {
            ".git", "__pycache__", ".cache",
            "node_modules"
        }
    ]

    for filename in files:
        lower = filename.lower()

        if (
            "humaneval" in lower
            or "human_eval" in lower
        ):
            path = os.path.join(root, filename)
            matches.append(path)

if matches:
    print("Found:")
    for path in matches:
        print(" ", path)
else:
    print("No HumanEval files found in current workspace.")

Searching current workspace for HumanEval files...

No HumanEval files found in current workspace.


In [ ]:
# ============================================
# PIPELINE B — RESTORE HUMANEVAL DATASET
# ============================================

import os
import json
from datasets import load_dataset

os.makedirs("tasks", exist_ok=True)

print("Loading HumanEval from Hugging Face...")

ds = load_dataset(
    "openai/openai_humaneval",
    split="test"
)

print("✓ HumanEval dataset loaded")
print("Tasks:", len(ds))

Loading HumanEval from Hugging Face...


README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

✓ HumanEval dataset loaded
Tasks: 164


In [ ]:
# ============================================
# PIPELINE B — SAVE HUMAN EVAL DATASET
# ============================================

import os
import json

os.makedirs("tasks", exist_ok=True)

humaneval_full = []

for item in ds:
    humaneval_full.append({
        "task_id": item["task_id"],
        "prompt": item["prompt"],
        "canonical_solution": item.get("canonical_solution", ""),
        "test": item.get("test", ""),
        "entry_point": item.get("entry_point", "")
    })

FULL_TASKS_PATH = "tasks/humaneval_full.json"

with open(FULL_TASKS_PATH, "w") as f:
    json.dump(humaneval_full, f, indent=2)

print("✓ HumanEval dataset saved")
print("Tasks:", len(humaneval_full))
print("File:", FULL_TASKS_PATH)

✓ HumanEval dataset saved
Tasks: 164
File: tasks/humaneval_full.json


In [ ]:
# ============================================
# PIPELINE B — CORE TASK VERIFICATION
# ============================================

CORE_TASK_IDS = [
    "HumanEval/53",
    "HumanEval/23",
    "HumanEval/45",
    "HumanEval/27",
    "HumanEval/15",
    "HumanEval/16",
    "HumanEval/98",
    "HumanEval/35",
    "HumanEval/13",
    "HumanEval/83",
    "HumanEval/50",
    "HumanEval/58",
    "HumanEval/89",
    "HumanEval/26",
    "HumanEval/154",
    "HumanEval/140",
    "HumanEval/111",
    "HumanEval/93",
    "HumanEval/56",
    "HumanEval/31",
    "HumanEval/7",
    "HumanEval/25",
    "HumanEval/132",
    "HumanEval/54",
    "HumanEval/20",
    "HumanEval/6",
    "HumanEval/41",
    "HumanEval/126",
    "HumanEval/43",
    "HumanEval/99",
]

# Build lookup from the restored 164-task dataset
task_by_id = {
    task["task_id"]: task
    for task in humaneval_full
}

# Check whether any required task is missing
missing = [
    task_id
    for task_id in CORE_TASK_IDS
    if task_id not in task_by_id
]

if missing:
    raise ValueError(
        f"Missing core tasks: {missing}"
    )

# Restore exact Pipeline A task order
core_tasks_b = [
    task_by_id[task_id]
    for task_id in CORE_TASK_IDS
]

# Verify uniqueness
unique_ids = {
    task["task_id"]
    for task in core_tasks_b
}

print("=" * 60)
print("PIPELINE B — CORE TASK VERIFICATION")
print("=" * 60)
print("Total core tasks:", len(core_tasks_b))
print("Unique task IDs:", len(unique_ids))
print("Missing tasks:", len(missing))

print("\nFirst 5:")
for task in core_tasks_b[:5]:
    print(task["task_id"])

print("\nLast 5:")
for task in core_tasks_b[-5:]:
    print(task["task_id"])

print("\n" + "=" * 60)

if (
    len(core_tasks_b) == 30
    and len(unique_ids) == 30
    and len(missing) == 0
):
    print("✓ EXACT 30 PIPELINE A TASKS RESTORED")
else:
    raise RuntimeError(
        "Core task verification failed."
    )

PIPELINE B — CORE TASK VERIFICATION
Total core tasks: 30
Unique task IDs: 30
Missing tasks: 0

First 5:
HumanEval/53
HumanEval/23
HumanEval/45
HumanEval/27
HumanEval/15

Last 5:
HumanEval/6
HumanEval/41
HumanEval/126
HumanEval/43
HumanEval/99

✓ EXACT 30 PIPELINE A TASKS RESTORED


In [ ]:
# ============================================================
# PIPELINE B — MULTI-AGENT FUNCTIONS
# ============================================================

import re
import time

TEMPERATURE = 0.01
MAX_TOKENS = 1024


# ------------------------------------------------------------
# LLM CALL
# ------------------------------------------------------------

def call_llm_b(system_prompt, user_prompt):

    start = time.perf_counter()

    response = client_b.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )

    elapsed = time.perf_counter() - start

    text = response.choices[0].message.content or ""

    tokens = 0
    if response.usage is not None:
        tokens = response.usage.total_tokens

    return {
        "text": text,
        "latency_sec": elapsed,
        "tokens": tokens
    }


# ------------------------------------------------------------
# CODE CLEANER
# ------------------------------------------------------------

def normalize_code(text):

    if not text:
        return ""

    text = text.strip()

    blocks = re.findall(
        r"```(?:python|py)?\s*(.*?)```",
        text,
        re.DOTALL | re.IGNORECASE
    )

    if blocks:
        return blocks[-1].strip()

    return text


# ------------------------------------------------------------
# AGENT 1 — GENERATOR
# ------------------------------------------------------------

def agent_generator(task):

    prompt = (
        "Solve this HumanEval problem.\n\n"
        "Return ONLY Python code.\n"
        "Preserve the exact function signature.\n"
        "Do not provide explanations.\n"
        "Do not use Markdown code fences.\n\n"
        "PROBLEM:\n"
        + task["prompt"]
    )

    result = call_llm_b(
        "You are Agent 1, the code generation agent.",
        prompt
    )

    result["code"] = normalize_code(result["text"])

    return result


# ------------------------------------------------------------
# AGENT 2 — REVIEWER
# ------------------------------------------------------------

def agent_reviewer(task, generated_code):

    prompt = (
        "Review this HumanEval solution.\n\n"
        "PROBLEM:\n"
        + task["prompt"]
        + "\n\nPROPOSED CODE:\n"
        + generated_code
        + "\n\n"
        "Check logical errors, edge cases, syntax, "
        "function signature, and specification compliance.\n"
        "Start with PASS or FAIL."
    )

    return call_llm_b(
        "You are Agent 2, a strict code reviewer.",
        prompt
    )


# ------------------------------------------------------------
# AGENT 3 — REFINER
# ------------------------------------------------------------

def agent_refiner(task, generated_code, review):

    prompt = (
        "Produce the final Python solution.\n\n"
        "ORIGINAL PROBLEM:\n"
        + task["prompt"]
        + "\n\nINITIAL SOLUTION:\n"
        + generated_code
        + "\n\nCODE REVIEW:\n"
        + review
        + "\n\n"
        "Correct any valid issues.\n"
        "Preserve the exact required function signature.\n"
        "Return ONLY Python code."
    )

    result = call_llm_b(
        "You are Agent 3, the final code refinement agent.",
        prompt
    )

    result["code"] = normalize_code(result["text"])

    return result


# ------------------------------------------------------------
# MULTI-AGENT PIPELINE
# ------------------------------------------------------------

def multi_agent_pipeline(task):

    start = time.perf_counter()

    generation = agent_generator(task)

    review = agent_reviewer(
        task,
        generation["code"]
    )

    refinement = agent_refiner(
        task,
        generation["code"],
        review["text"]
    )

    total_latency = time.perf_counter() - start

    total_tokens = (
        (generation["tokens"] or 0)
        + (review["tokens"] or 0)
        + (refinement["tokens"] or 0)
    )

    return {
        "task_id": task["task_id"],
        "prompt": task["prompt"],
        "initial_code": generation["code"],
        "review": review["text"],
        "final_code": refinement["code"],
        "generation_latency_sec": generation["latency_sec"],
        "review_latency_sec": review["latency_sec"],
        "refinement_latency_sec": refinement["latency_sec"],
        "total_latency_sec": total_latency,
        "total_tokens": total_tokens,
        "error": None
    }


# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print("=" * 60)
print("PIPELINE B — FUNCTION VERIFICATION")
print("=" * 60)

print("call_llm_b:", callable(call_llm_b))
print("agent_generator:", callable(agent_generator))
print("agent_reviewer:", callable(agent_reviewer))
print("agent_refiner:", callable(agent_refiner))
print("multi_agent_pipeline:", callable(multi_agent_pipeline))

if not callable(multi_agent_pipeline):
    raise RuntimeError("Pipeline B definition failed.")

print("=" * 60)
print("✓ PIPELINE B FUNCTIONS READY")
print("=" * 60)

PIPELINE B — FUNCTION VERIFICATION
call_llm_b: True
agent_generator: True
agent_reviewer: True
agent_refiner: True
multi_agent_pipeline: True
✓ PIPELINE B FUNCTIONS READY


In [ ]:
# ============================================================
# PIPELINE B — SINGLE TASK TEST
# HumanEval/53 ONLY
# ============================================================

test_task_b = core_tasks_b[0]

print("=" * 60)
print("PIPELINE B — SINGLE TASK TEST")
print("=" * 60)
print("Task:", test_task_b["task_id"])

try:
    test_result_b = multi_agent_pipeline(test_task_b)

    print("\n✓ MULTI-AGENT EXECUTION COMPLETED")

    print("\n" + "-" * 60)
    print("AGENT 1 — GENERATED CODE")
    print("-" * 60)
    print(test_result_b["initial_code"])

    print("\n" + "-" * 60)
    print("AGENT 2 — REVIEW")
    print("-" * 60)
    print(test_result_b["review"])

    print("\n" + "-" * 60)
    print("AGENT 3 — FINAL CODE")
    print("-" * 60)
    print(test_result_b["final_code"])

    print("\n" + "-" * 60)
    print("METRICS")
    print("-" * 60)

    print(
        "Generation:",
        round(test_result_b["generation_latency_sec"], 2),
        "s"
    )

    print(
        "Review:",
        round(test_result_b["review_latency_sec"], 2),
        "s"
    )

    print(
        "Refinement:",
        round(test_result_b["refinement_latency_sec"], 2),
        "s"
    )

    print(
        "Total:",
        round(test_result_b["total_latency_sec"], 2),
        "s"
    )

    print("Total tokens:", test_result_b["total_tokens"])

except Exception as e:
    print("\n✗ MULTI-AGENT TEST FAILED")
    print("Error:", type(e).__name__)
    print(str(e))
    raise

PIPELINE B — SINGLE TASK TEST
Task: HumanEval/53

✓ MULTI-AGENT EXECUTION COMPLETED

------------------------------------------------------------
AGENT 1 — GENERATED CODE
------------------------------------------------------------
def add(x: int, y: int) -> int:
    return x + y

------------------------------------------------------------
AGENT 2 — REVIEW
------------------------------------------------------------
FAIL

The proposed code is missing the docstring examples. The original problem statement includes two doctest examples, but the proposed code does not include them. The docstring should include these examples to ensure the function behaves as expected.

Additionally, the proposed code does not handle edge cases such as:

* What happens when the inputs are not integers? The function signature suggests that it only accepts integers, but the implementation does not check for this.
* What happens when the inputs are very large numbers? The function does not handle potential o

In [ ]:
# ============================================================
# PIPELINE B — RESEARCH-SAFE REVIEW + REFINEMENT
# ============================================================

def agent_reviewer(task, generated_code):

    prompt = (
        "Review the proposed solution ONLY against the original "
        "HumanEval problem specification.\n\n"

        "IMPORTANT RULES:\n"
        "1. Do not invent requirements.\n"
        "2. Do not require docstrings unless the problem explicitly "
        "requires them.\n"
        "3. Do not require type checking unless explicitly required.\n"
        "4. Do not require overflow handling unless explicitly required.\n"
        "5. Do not require examples unless explicitly required.\n"
        "6. Do not criticize valid Python behavior merely because "
        "additional validation could be added.\n"
        "7. Focus on whether the implementation satisfies the stated "
        "functional requirements and handles relevant specified cases.\n"
        "8. If the solution is functionally correct, mark it PASS.\n\n"

        "ORIGINAL HUMAN EVAL PROBLEM:\n"
        + task["prompt"]
        + "\n\nPROPOSED SOLUTION:\n"
        + generated_code
        + "\n\n"

        "Start your response with exactly PASS or FAIL.\n"
        "If FAIL, identify only genuine specification violations "
        "or meaningful correctness problems."
    )

    return call_llm_b(
        "You are Agent 2, a rigorous HumanEval code reviewer. "
        "Judge correctness strictly according to the supplied "
        "problem specification.",
        prompt
    )


def agent_refiner(task, generated_code, review):

    prompt = (
        "Produce the final Python solution for the HumanEval problem.\n\n"

        "IMPORTANT RULES:\n"
        "1. Follow the original HumanEval specification.\n"
        "2. Apply only genuine corrections supported by the "
        "specification or review.\n"
        "3. Do not add unnecessary validation.\n"
        "4. Do not add arbitrary type restrictions.\n"
        "5. Do not add artificial overflow checks.\n"
        "6. Do not add unnecessary exceptions.\n"
        "7. Do not add documentation merely because the reviewer "
        "suggested it unless documentation is required.\n"
        "8. Preserve the required function signature.\n"
        "9. Return ONLY Python code.\n\n"

        "ORIGINAL PROBLEM:\n"
        + task["prompt"]
        + "\n\nINITIAL SOLUTION:\n"
        + generated_code
        + "\n\nREVIEW:\n"
        + review
    )

    result = call_llm_b(
        "You are Agent 3, the final code refinement agent. "
        "Produce the simplest correct solution that satisfies "
        "the original HumanEval specification.",
        prompt
    )

    result["code"] = normalize_code(result["text"])

    return result


def multi_agent_pipeline(task):

    pipeline_start = time.perf_counter()

    generation = agent_generator(task)

    review = agent_reviewer(
        task,
        generation["code"]
    )

    refinement = agent_refiner(
        task,
        generation["code"],
        review["text"]
    )

    total_latency = time.perf_counter() - pipeline_start

    total_tokens = (
        (generation["tokens"] or 0)
        + (review["tokens"] or 0)
        + (refinement["tokens"] or 0)
    )

    return {
        "task_id": task["task_id"],
        "prompt": task["prompt"],
        "initial_code": generation["code"],
        "review": review["text"],
        "final_code": refinement["code"],
        "generation_latency_sec": generation["latency_sec"],
        "review_latency_sec": review["latency_sec"],
        "refinement_latency_sec": refinement["latency_sec"],
        "total_latency_sec": total_latency,
        "total_tokens": total_tokens,
        "error": None
    }


print("=" * 60)
print("✓ RESEARCH-SAFE PIPELINE B FUNCTIONS UPDATED")
print("=" * 60)

✓ RESEARCH-SAFE PIPELINE B FUNCTIONS UPDATED


In [ ]:
# ============================================================
# PIPELINE B — FINAL SMOKE TEST
# ============================================================

test_task_b = core_tasks_b[0]

print("=" * 60)
print("PIPELINE B — FINAL SMOKE TEST")
print("=" * 60)
print("Task:", test_task_b["task_id"])

test_result_b = multi_agent_pipeline(test_task_b)

print("\n✓ MULTI-AGENT EXECUTION COMPLETED")

print("\n" + "-" * 60)
print("AGENT 1 — GENERATED CODE")
print("-" * 60)
print(test_result_b["initial_code"])

print("\n" + "-" * 60)
print("AGENT 2 — REVIEW")
print("-" * 60)
print(test_result_b["review"])

print("\n" + "-" * 60)
print("AGENT 3 — FINAL CODE")
print("-" * 60)
print(test_result_b["final_code"])

print("\n" + "-" * 60)
print("METRICS")
print("-" * 60)

print("Generation:",
      round(test_result_b["generation_latency_sec"], 2), "s")

print("Review:",
      round(test_result_b["review_latency_sec"], 2), "s")

print("Refinement:",
      round(test_result_b["refinement_latency_sec"], 2), "s")

print("Total:",
      round(test_result_b["total_latency_sec"], 2), "s")

print("Total tokens:",
      test_result_b["total_tokens"])

PIPELINE B — FINAL SMOKE TEST
Task: HumanEval/53

✓ MULTI-AGENT EXECUTION COMPLETED

------------------------------------------------------------
AGENT 1 — GENERATED CODE
------------------------------------------------------------
def add(x: int, y: int) -> int:
    return x + y

------------------------------------------------------------
AGENT 2 — REVIEW
------------------------------------------------------------
PASS. 

The proposed solution correctly implements the required functionality of adding two numbers and returns the result. It also matches the specified type hints and the return type. The docstring is also correctly implemented as per the problem specification.

------------------------------------------------------------
AGENT 3 — FINAL CODE
------------------------------------------------------------
def add(x: int, y: int) -> int:
    """Add two numbers x and y
    >>> add(2, 3)
    5
    >>> add(5, 7)
    12
    """
    return x + y

---------------------------------

In [ ]:
# ============================================================
# PIPELINE B — CHECKPOINTED 30-TASK EXPERIMENT
# ============================================================

import os
import json
import time

RESULTS_DIR = "results"
CHECKPOINT_PATH = os.path.join(
    RESULTS_DIR,
    "pipeline_b_checkpoint.json"
)

os.makedirs(RESULTS_DIR, exist_ok=True)


# ------------------------------------------------------------
# LOAD EXISTING CHECKPOINT
# ------------------------------------------------------------

if os.path.exists(CHECKPOINT_PATH):

    with open(CHECKPOINT_PATH, "r") as f:
        pipeline_b_results = json.load(f)

    print("✓ Existing Pipeline B checkpoint loaded")
    print("Existing records:", len(pipeline_b_results))

else:

    pipeline_b_results = []

    print("✓ Starting new Pipeline B experiment")


# ------------------------------------------------------------
# BUILD COMPLETED-ID SET
# ------------------------------------------------------------

completed_ids = {
    record["task_id"]
    for record in pipeline_b_results
    if record.get("error") is None
}

print("Completed task IDs:", len(completed_ids))


# ------------------------------------------------------------
# RUN EXACT 30 TASKS
# ------------------------------------------------------------

print("=" * 60)
print("PIPELINE B — START")
print("=" * 60)

print("Model:", MODEL)
print("Tasks:", len(core_tasks_b))
print("Already completed:", len(completed_ids))
print("=" * 60)


for index, task in enumerate(core_tasks_b, start=1):

    task_id = task["task_id"]

    # --------------------------------------------------------
    # NEVER RE-RUN COMPLETED TASKS
    # --------------------------------------------------------

    if task_id in completed_ids:

        print(
            f"[{index}/30] {task_id} | "
            "SKIPPED — already completed"
        )

        continue


    print(
        f"\n[{index}/30] {task_id}"
    )

    start_time = time.perf_counter()


    try:

        # ----------------------------------------------------
        # MULTI-AGENT EXECUTION
        # ----------------------------------------------------

        result = multi_agent_pipeline(task)

        result["run_index"] = index
        result["smoke_test"] = False
        result["provider"] = getattr(
            client_b,
            "provider",
            None
        )
        result["model"] = MODEL

        result["experiment_timestamp"] = (
            time.strftime("%Y-%m-%d %H:%M:%S")
        )


        # ----------------------------------------------------
        # ADD RESULT
        # ----------------------------------------------------

        pipeline_b_results.append(result)

        completed_ids.add(task_id)


        # ----------------------------------------------------
        # SAVE IMMEDIATELY
        # ----------------------------------------------------

        with open(CHECKPOINT_PATH, "w") as f:
            json.dump(
                pipeline_b_results,
                f,
                indent=2
            )


        elapsed = time.perf_counter() - start_time

        print(
            f"✓ SUCCESS | "
            f"{elapsed:.2f}s | "
            f"{result['total_tokens']} tokens"
        )

        print(
            f"Checkpoint saved | "
            f"{len(pipeline_b_results)} records"
        )


    except Exception as e:

        elapsed = time.perf_counter() - start_time

        error_record = {
            "task_id": task_id,
            "run_index": index,
            "model": MODEL,
            "provider": getattr(
                client_b,
                "provider",
                None
            ),
            "error": str(e),
            "error_type": type(e).__name__,
            "elapsed_sec": elapsed,
            "smoke_test": False,
            "experiment_timestamp": (
                time.strftime("%Y-%m-%d %H:%M:%S")
            )
        }


        print(
            f"✗ ERROR | "
            f"{type(e).__name__}: {e}"
        )


        # ----------------------------------------------------
        # CREDIT / AUTH / PROVIDER ERROR
        # ----------------------------------------------------

        error_text = str(e).lower()

        credit_error = (
            "402" in error_text
            or "payment required" in error_text
            or "depleted" in error_text
            or "credits" in error_text
        )

        if credit_error:

            print("\n" + "=" * 60)
            print("⚠ HUGGING FACE CREDIT LIMIT REACHED")
            print("=" * 60)
            print(
                "Stopping safely."
            )
            print(
                "Successful results are already checkpointed."
            )
            print(
                "Restarting this cell will resume "
                "without rerunning completed tasks."
            )
            print("=" * 60)

            break


        # ----------------------------------------------------
        # OTHER ERROR
        # ----------------------------------------------------

        print(
            "Non-credit error encountered."
        )

        print(
            "The task was NOT added as a valid result."
        )

        # Continue to next task
        continue


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

successful_results = [
    r
    for r in pipeline_b_results
    if r.get("error") is None
]

successful_ids = {
    r["task_id"]
    for r in successful_results
}

failed_ids = [
    task["task_id"]
    for task in core_tasks_b
    if task["task_id"] not in successful_ids
]


print("\n" + "=" * 60)
print("PIPELINE B — RUN STATUS")
print("=" * 60)

print("Total target tasks:", len(core_tasks_b))
print("Successful:", len(successful_results))
print("Remaining:", len(failed_ids))

if failed_ids:

    print("\nRemaining tasks:")

    for task_id in failed_ids:
        print(task_id)

else:

    print("\n✓ ALL 30 TASKS COMPLETED")

print("=" * 60)
print("Checkpoint:", CHECKPOINT_PATH)
print("=" * 60)

✓ Starting new Pipeline B experiment
Completed task IDs: 0
PIPELINE B — START
Model: meta-llama/Llama-3.1-8B-Instruct
Tasks: 30
Already completed: 0

[1/30] HumanEval/53
✓ SUCCESS | 4.61s | 726 tokens
Checkpoint saved | 1 records

[2/30] HumanEval/23
✓ SUCCESS | 5.34s | 725 tokens
Checkpoint saved | 2 records

[3/30] HumanEval/45
✓ SUCCESS | 6.75s | 767 tokens
Checkpoint saved | 3 records

[4/30] HumanEval/27
✓ SUCCESS | 5.46s | 768 tokens
Checkpoint saved | 4 records

[5/30] HumanEval/15
✓ SUCCESS | 5.32s | 826 tokens
Checkpoint saved | 5 records

[6/30] HumanEval/16
✗ ERROR | HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a8b0e11-3d130aa5556b2d0c7af6c317;fa5b5c65-415c-4e4d-bb48-5cdb37f5ab58)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alterna

In [ ]:
# ============================================================
# PIPELINE B — CONTINUE REMAINING TASKS
# ============================================================

import os
import json
import time

CHECKPOINT_PATH = "results/pipeline_b_checkpoint.json"

# ------------------------------------------------------------
# LOAD EXISTING CHECKPOINT
# ------------------------------------------------------------

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"{CHECKPOINT_PATH} not found. "
        "Make sure the previous Pipeline B run was saved."
    )

with open(CHECKPOINT_PATH, "r") as f:
    pipeline_b_results = json.load(f)

print("=" * 60)
print("PIPELINE B — CONTINUATION")
print("=" * 60)
print("Checkpoint records:", len(pipeline_b_results))


# ------------------------------------------------------------
# FIND COMPLETED TASKS
# ------------------------------------------------------------

completed_ids = {
    r["task_id"]
    for r in pipeline_b_results
    if r.get("error") is None
}

remaining_tasks = [
    task
    for task in core_tasks_b
    if task["task_id"] not in completed_ids
]

print("Completed:", len(completed_ids))
print("Remaining:", len(remaining_tasks))
print("=" * 60)


# ------------------------------------------------------------
# RUN ONLY REMAINING TASKS
# ------------------------------------------------------------

for run_number, task in enumerate(
    remaining_tasks,
    start=1
):

    task_id = task["task_id"]

    print(
        f"\n[{run_number}/{len(remaining_tasks)}] "
        f"{task_id}"
    )

    start_time = time.perf_counter()

    try:

        # ----------------------------------------------------
        # MULTI-AGENT PIPELINE
        # ----------------------------------------------------

        result = multi_agent_pipeline(task)

        result["smoke_test"] = False
        result["model"] = MODEL
        result["provider"] = getattr(
            client_b,
            "provider",
            None
        )
        result["continuation_run"] = True
        result["experiment_timestamp"] = (
            time.strftime("%Y-%m-%d %H:%M:%S")
        )

        # ----------------------------------------------------
        # SAVE SUCCESSFUL RESULT
        # ----------------------------------------------------

        pipeline_b_results.append(result)
        completed_ids.add(task_id)

        with open(CHECKPOINT_PATH, "w") as f:
            json.dump(
                pipeline_b_results,
                f,
                indent=2
            )

        elapsed = time.perf_counter() - start_time

        print(
            f"✓ SUCCESS | "
            f"{elapsed:.2f}s | "
            f"{result['total_tokens']} tokens"
        )

        print(
            "Checkpoint saved:",
            len(pipeline_b_results),
            "records"
        )

    except Exception as e:

        elapsed = time.perf_counter() - start_time
        error_text = str(e)

        print(
            f"✗ ERROR: "
            f"{type(e).__name__}: {error_text}"
        )

        # ----------------------------------------------------
        # HUGGING FACE CREDIT ERROR
        # ----------------------------------------------------

        error_lower = error_text.lower()

        credit_error = (
            "402" in error_lower
            or "payment required" in error_lower
            or "depleted" in error_lower
            or "credits" in error_lower
        )

        if credit_error:

            print("\n" + "=" * 60)
            print("⚠ HUGGING FACE CREDIT LIMIT REACHED")
            print("=" * 60)
            print(
                "Completed results are safely preserved."
            )
            print(
                "Checkpoint records:",
                len(pipeline_b_results)
            )
            print(
                "Run this same continuation cell again "
                "when credits are available."
            )
            print("=" * 60)

            break

        print("Continuing to next task...")


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

successful_results = [
    r
    for r in pipeline_b_results
    if r.get("error") is None
]

successful_ids = {
    r["task_id"]
    for r in successful_results
}

remaining_ids = [
    task["task_id"]
    for task in core_tasks_b
    if task["task_id"] not in successful_ids
]


print("\n" + "=" * 60)
print("PIPELINE B — CONTINUATION COMPLETE")
print("=" * 60)

print(
    "Total valid results:",
    len(successful_results)
)

print(
    "Remaining:",
    len(remaining_ids)
)

if remaining_ids:

    print("\nRemaining tasks:")

    for task_id in remaining_ids:
        print(task_id)

else:

    print("\n✓ ALL 30 PIPELINE B TASKS COMPLETED")

print(
    "\nCheckpoint:",
    CHECKPOINT_PATH
)

print("=" * 60)

PIPELINE B — CONTINUATION
Checkpoint records: 5
Completed: 5
Remaining: 25

[1/25] HumanEval/16
✓ SUCCESS | 4.20s | 792 tokens
Checkpoint saved: 6 records

[2/25] HumanEval/98
✓ SUCCESS | 5.03s | 1034 tokens
Checkpoint saved: 7 records

[3/25] HumanEval/35
✓ SUCCESS | 6.52s | 850 tokens
Checkpoint saved: 8 records

[4/25] HumanEval/13
✓ SUCCESS | 4.36s | 881 tokens
Checkpoint saved: 9 records

[5/25] HumanEval/83
✗ ERROR: HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a8b0ece-0ff32faa15ca3c4d0c6a0204;0d266ede-8abe-41cf-80ea-cdd89d374cc0)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

⚠ HUGGING FACE CREDIT LIMIT REACHED
Completed results are safely preserved.
Checkpoint records: 9
Run

In [ ]:
import json

PATH = "results/pipeline_b_checkpoint.json"

with open(PATH) as f:
    results = json.load(f)

done = {r["task_id"] for r in results if r.get("error") is None}
remaining = [t for t in core_tasks_b if t["task_id"] not in done]

print(f"Already done: {len(done)} | Remaining: {len(remaining)}")

for task in remaining:
    try:
        r = multi_agent_pipeline(task)
        r["smoke_test"] = False
        r["model"] = MODEL
        r["provider"] = getattr(client_b, "provider", None)

        results.append(r)

        with open(PATH, "w") as f:
            json.dump(results, f, indent=2)

        print(f"✓ {task['task_id']} saved | Total: {len(results)}")

    except Exception as e:
        print(f"✗ {task['task_id']}: {e}")

        if "402" in str(e) or "credit" in str(e).lower():
            print("⚠ Credits exhausted — stopping safely.")
            break

print("\nValid results:", len([
    r for r in results if r.get("error") is None
]))
print("Remaining:", 30 - len([
    r for r in results if r.get("error") is None
]))

Already done: 9 | Remaining: 21
✓ HumanEval/83 saved | Total: 10
✓ HumanEval/50 saved | Total: 11
✓ HumanEval/58 saved | Total: 12
✗ HumanEval/89: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a8b0f08-4a47c78a70d501c30f57c8db;03cee405-93d6-4e36-b641-b8b26ab18a65)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
⚠ Credits exhausted — stopping safely.

Valid results: 12
Remaining: 18


In [ ]:
import shutil

shutil.copy(
    "results/pipeline_b_checkpoint.json",
    "results/pipeline_b_backup_12.json"
)

print("✓ Pipeline B backup saved")
print("Valid results: 12")

✓ Pipeline B backup saved
Valid results: 12


In [ ]:
import json

PATH = "results/pipeline_b_checkpoint.json"

with open(PATH) as f:
    results = json.load(f)

done = {r["task_id"] for r in results if r.get("error") is None}
remaining = [t for t in core_tasks_b if t["task_id"] not in done]

print(f"Already done: {len(done)} | Remaining: {len(remaining)}")

for task in remaining:
    try:
        r = multi_agent_pipeline(task)

        r["smoke_test"] = False
        r["model"] = MODEL
        r["provider"] = getattr(client_b, "provider", None)

        results.append(r)

        with open(PATH, "w") as f:
            json.dump(results, f, indent=2)

        print(f"✓ {task['task_id']} saved | Total: {len(results)}")

    except Exception as e:
        print(f"✗ {task['task_id']}: {e}")

        if "402" in str(e) or "credit" in str(e).lower():
            print("⚠ Credits exhausted — stopping safely.")
            break

print(
    "\nValid results:",
    len([r for r in results if r.get("error") is None])
)

print(
    "Remaining:",
    30 - len([r for r in results if r.get("error") is None])
)

Already done: 12 | Remaining: 18
✓ HumanEval/89 saved | Total: 13
✓ HumanEval/26 saved | Total: 14
✓ HumanEval/154 saved | Total: 15
✗ HumanEval/140: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a8b0f8b-4ce42ce525056d952bc4a4b6;5d1f0600-9ec8-4d3f-a7c8-6e8f7013eb12)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
⚠ Credits exhausted — stopping safely.

Valid results: 15
Remaining: 15


In [ ]:
# ============================================================
# PIPELINE B — CONTINUE + SAVE
# ============================================================

import json

PATH = "results/pipeline_b_checkpoint.json"

with open(PATH) as f:
    results = json.load(f)

done = {
    r["task_id"]
    for r in results
    if r.get("error") is None
}

remaining = [
    t for t in core_tasks_b
    if t["task_id"] not in done
]

print(f"Already done: {len(done)} | Remaining: {len(remaining)}")

for task in remaining:
    try:
        r = multi_agent_pipeline(task)

        r["smoke_test"] = False
        r["model"] = MODEL
        r["provider"] = getattr(client_b, "provider", None)

        results.append(r)

        # Save immediately after every successful task
        with open(PATH, "w") as f:
            json.dump(results, f, indent=2)

        print(
            f"✓ {task['task_id']} saved | "
            f"Total: {len(results)}"
        )

    except Exception as e:
        print(f"✗ {task['task_id']}: {e}")

        if (
            "402" in str(e)
            or "credit" in str(e).lower()
            or "depleted" in str(e).lower()
        ):
            print("⚠ Credits exhausted — stopping safely.")
            break

valid = [
    r for r in results
    if r.get("error") is None
]

print("\n" + "=" * 60)
print("PIPELINE B STATUS")
print("=" * 60)
print("Valid results:", len(valid))
print("Remaining:", 30 - len(valid))
print("Checkpoint:", PATH)
print("=" * 60)

Already done: 15 | Remaining: 15
✓ HumanEval/140 saved | Total: 16
✓ HumanEval/111 saved | Total: 17
✓ HumanEval/93 saved | Total: 18
✗ HumanEval/56: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a8b0fc4-477d8fa12ecea894123268b6;7998e6c9-37b0-411a-b691-b28f0133a0a4)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
⚠ Credits exhausted — stopping safely.

PIPELINE B STATUS
Valid results: 18
Remaining: 12
Checkpoint: results/pipeline_b_checkpoint.json


In [ ]:
import json

PATH = "results/pipeline_b_checkpoint.json"

with open(PATH) as f:
    results = json.load(f)

done = {r["task_id"] for r in results if r.get("error") is None}
remaining = [t for t in core_tasks_b if t["task_id"] not in done]

print(f"Already done: {len(done)} | Remaining: {len(remaining)}")

for task in remaining:
    try:
        r = multi_agent_pipeline(task)

        r["smoke_test"] = False
        r["model"] = MODEL
        r["provider"] = getattr(client_b, "provider", None)

        results.append(r)

        # Save immediately
        with open(PATH, "w") as f:
            json.dump(results, f, indent=2)

        print(f"✓ {task['task_id']} saved | Total: {len(results)}")

    except Exception as e:
        print(f"✗ {task['task_id']}: {e}")

        if "402" in str(e) or "credit" in str(e).lower():
            print("⚠ Credits exhausted — stopping safely.")
            break

valid = [r for r in results if r.get("error") is None]

print("\n" + "=" * 60)
print("PIPELINE B STATUS")
print("=" * 60)
print("Valid results:", len(valid))
print("Remaining:", 30 - len(valid))
print("Checkpoint:", PATH)
print("=" * 60)

Already done: 18 | Remaining: 12
✓ HumanEval/56 saved | Total: 19
✓ HumanEval/31 saved | Total: 20
✓ HumanEval/7 saved | Total: 21
✗ HumanEval/25: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a8b0ff8-7696d2176dede471662a3b8d;28341c69-f2f7-4ae6-bbe5-586fb963f92c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
⚠ Credits exhausted — stopping safely.

PIPELINE B STATUS
Valid results: 21
Remaining: 9
Checkpoint: results/pipeline_b_checkpoint.json


In [ ]:
import json

PATH = "results/pipeline_b_checkpoint.json"

with open(PATH) as f:
    results = json.load(f)

done = {r["task_id"] for r in results if r.get("error") is None}
remaining = [t for t in core_tasks_b if t["task_id"] not in done]

print(f"Already done: {len(done)} | Remaining: {len(remaining)}")

for task in remaining:
    try:
        r = multi_agent_pipeline(task)
        r["smoke_test"] = False
        r["model"] = MODEL
        r["provider"] = getattr(client_b, "provider", None)

        results.append(r)

        # Save after every successful task
        with open(PATH, "w") as f:
            json.dump(results, f, indent=2)

        print(f"✓ {task['task_id']} saved | Total: {len(results)}")

    except Exception as e:
        print(f"✗ {task['task_id']}: {e}")

        if "402" in str(e) or "credit" in str(e).lower():
            print("⚠ Credits exhausted — stopping safely.")
            break

valid = [r for r in results if r.get("error") is None]

print("\n" + "=" * 60)
print("PIPELINE B STATUS")
print("=" * 60)
print("Valid results:", len(valid))
print("Remaining:", 30 - len(valid))
print("Checkpoint:", PATH)
print("=" * 60)

Already done: 21 | Remaining: 9
✓ HumanEval/25 saved | Total: 22
✓ HumanEval/132 saved | Total: 23
✓ HumanEval/54 saved | Total: 24
✓ HumanEval/20 saved | Total: 25
✓ HumanEval/6 saved | Total: 26
✗ HumanEval/41: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a8b110b-52d3abbe1f38f9145f28ca74;0c6a63b1-8aff-483b-a6a6-3a221df3e0fd)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
⚠ Credits exhausted — stopping safely.

PIPELINE B STATUS
Valid results: 26
Remaining: 4
Checkpoint: results/pipeline_b_checkpoint.json


In [ ]:
import json

PATH = "results/pipeline_b_checkpoint.json"

with open(PATH) as f:
    results = json.load(f)

done = {r["task_id"] for r in results if r.get("error") is None}
remaining = [t for t in core_tasks_b if t["task_id"] not in done]

print(f"Already done: {len(done)} | Remaining: {len(remaining)}")

for task in remaining:
    try:
        r = multi_agent_pipeline(task)
        r["smoke_test"] = False
        r["model"] = MODEL
        r["provider"] = getattr(client_b, "provider", None)

        results.append(r)

        # Save immediately
        with open(PATH, "w") as f:
            json.dump(results, f, indent=2)

        print(f"✓ {task['task_id']} saved | Total: {len(results)}")

    except Exception as e:
        print(f"✗ {task['task_id']}: {e}")

        if (
            "402" in str(e)
            or "credit" in str(e).lower()
            or "depleted" in str(e).lower()
        ):
            print("⚠ Credits exhausted — stopping safely.")
            break

valid = [r for r in results if r.get("error") is None]

print("\n" + "=" * 60)
print("PIPELINE B STATUS")
print("=" * 60)
print("Valid results:", len(valid))
print("Remaining:", 30 - len(valid))
print("Checkpoint:", PATH)
print("=" * 60)

Already done: 27 | Remaining: 3
✓ HumanEval/41 saved | Total: 28
✓ HumanEval/43 saved | Total: 29
✗ HumanEval/99: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a8b11c3-7cf1f70376acc7d16337a7f3;379f6788-95f6-4093-83f5-1edb3d9568e2)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
⚠ Credits exhausted — stopping safely.

PIPELINE B STATUS
Valid results: 29
Remaining: 1
Checkpoint: results/pipeline_b_checkpoint.json


In [ ]:
import json

PATH = "results/pipeline_b_checkpoint.json"

with open(PATH) as f:
    results = json.load(f)

done = {r["task_id"] for r in results if r.get("error") is None}
task = next(t for t in core_tasks_b if t["task_id"] not in done)

print("Running:", task["task_id"])

try:
    r = multi_agent_pipeline(task)
    r["smoke_test"] = False
    r["model"] = MODEL
    r["provider"] = getattr(client_b, "provider", None)

    results.append(r)

    with open(PATH, "w") as f:
        json.dump(results, f, indent=2)

    print("✓ Saved:", task["task_id"])

except Exception as e:
    print("✗ Error:", e)

print(
    "Valid results:",
    len([r for r in results if r.get("error") is None])
)

Running: HumanEval/99
✓ Saved: HumanEval/99
Valid results: 30


In [ ]:
import json
import os

CHECKPOINT = "results/pipeline_b_checkpoint.json"
FINAL = "results/pipeline_b_final_30.json"

with open(CHECKPOINT) as f:
    results = json.load(f)

valid = [r for r in results if r.get("error") is None]
task_ids = [r["task_id"] for r in valid]

expected_ids = [t["task_id"] for t in core_tasks_b]

print("=" * 60)
print("PIPELINE B — FINAL VALIDATION")
print("=" * 60)

print("Valid results:", len(valid))
print("Unique task IDs:", len(set(task_ids)))
print("Expected tasks:", len(expected_ids))

missing = sorted(set(expected_ids) - set(task_ids))
extra = sorted(set(task_ids) - set(expected_ids))
duplicates = len(task_ids) - len(set(task_ids))

print("Missing:", missing)
print("Unexpected:", extra)
print("Duplicate count:", duplicates)

assert len(valid) == 30
assert len(set(task_ids)) == 30
assert not missing
assert not extra
assert duplicates == 0

# Save ONLY the 30 valid experiment records
with open(FINAL, "w") as f:
    json.dump(valid, f, indent=2)

print("\n" + "=" * 60)
print("✓ OFFICIAL PIPELINE B DATASET SAVED")
print("=" * 60)
print("File:", FINAL)
print("Records:", len(valid))
print("Unique tasks:", len(set(task_ids)))
print("=" * 60)

PIPELINE B — FINAL VALIDATION
Valid results: 30
Unique task IDs: 30
Expected tasks: 30
Missing: []
Unexpected: []
Duplicate count: 0

✓ OFFICIAL PIPELINE B DATASET SAVED
File: results/pipeline_b_final_30.json
Records: 30
Unique tasks: 30


In [ ]:
from google.colab import files

files.download("results/pipeline_b_final_30.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>